# NFL Draft Curves
**Author:** Schuckers

## NFL Draft Pick Value Curves

So this file is all about making a draft pick value curve/chart (DPVC).  And covering some 
of the basic techniques for fitting a smooth valuation curve to these data.  Note
that almost all of these methods apply to different sports.  

There are a couple of choices to be made when you decide to make a DPVC.  The first
choice is what to have as your target/response.  There are basically three choices
that have been historically used and one choice that is relatively new.  Historically,
the options are 1) annual value of a second contract, 2) performance over some
period (usually over length of team control of first contract), 3) value of historical
trades.  The first in the list above is market driven and economic, while the second
is more performance based.  The response/target for these two methods is straightforward
though the contract data can be more difficult to collect.  The third method is
somewhat harder to generate in some sports because top picks are rarely traded.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import cr, dmatrix
from plotnine import (
    ggplot, aes, geom_point, geom_line, stat_smooth, scale_x_continuous,
    labs, theme_minimal, theme, element_text, element_blank
)
import warnings
warnings.filterwarnings('ignore')

### Load in the draft data

In [ ]:
# Load draft picks for 2015-2021 from nflverse data repository
url = "https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.csv"
draft_picks_raw = pd.read_csv(url)

cols = [
    'season', 'round', 'pick', 'team', 'pfr_player_id', 'pfr_player_name', 
    'position', 'category', 'side', 'allpro', 'probowls', 'seasons_started', 
    'w_av', 'car_av', 'dr_av'
]

draft_picks = (
    draft_picks_raw[draft_picks_raw['season'].between(2015, 2021)]
    [cols]
    .dropna(subset=['dr_av'])
    .copy()
)

### Loess smoothers
Here we are going to use local smoothing via LOESS

In [ ]:
# Plot Career_AV vs Overall_Pick with LOESS curve (span = 0.75)
(
    ggplot(draft_picks, aes(x='pick', y='dr_av'))
    + geom_point(alpha=0.35, color="#2b5c8f", size=2)
    + stat_smooth(
        method="loess",
        span=0.75,         # Adjust span (0.3 to 1.0) to control smoothing
        color="blue",
        fill="gray",       # Confidence interval band color
        se=True,
        size=1.2
    )
    + scale_x_continuous(breaks=range(0, 261, 32))
    + labs(
        title="NFL Draft Picks: Career Approximate Value (AV) vs. Overall Pick",
        subtitle="Draft Classes 2015–2021 (Rounds 1–7), span=0.75",
        x="Overall Pick Number",
        y="Career Approximate Value (AV)",
        caption="Data: nflverse / Pro-Football-Reference"
    )
    + theme_minimal(base_size=13)
    + theme(
        plot_title=element_text(weight="bold"),
        panel_grid_minor=element_blank()
    )
)

Change the size of the window for LOESS to be smaller.

In [ ]:
# Plot Career_AV vs Overall_Pick with LOESS curve (span = 0.5)
(
    ggplot(draft_picks, aes(x='pick', y='dr_av'))
    + geom_point(alpha=0.35, color="#2b5c8f", size=2)
    + stat_smooth(
        method="loess",
        span=0.5,          # Adjust span (0.3 to 1.0) to control smoothing
        color="blue",
        fill="gray",       # Confidence interval band color
        se=True,
        size=1.2
    )
    + scale_x_continuous(breaks=range(0, 261, 32))
    + labs(
        title="NFL Draft Picks: Career Approximate Value (AV) vs. Overall Pick",
        subtitle="Draft Classes 2015–2021 (Rounds 1–7), span=0.5",
        x="Overall Pick Number",
        y="Career Approximate Value (AV)",
        caption="Data: nflverse / Pro-Football-Reference"
    )
    + theme_minimal(base_size=13)
    + theme(
        plot_title=element_text(weight="bold"),
        panel_grid_minor=element_blank()
    )
)

Change the span again to be much smaller

In [ ]:
# Plot Career_AV vs Overall_Pick with LOESS curve (span = 0.15)
(
    ggplot(draft_picks, aes(x='pick', y='dr_av'))
    + geom_point(alpha=0.35, color="#2b5c8f", size=2)
    + stat_smooth(
        method="loess",
        span=0.15,         # Adjust span (0.3 to 1.0) to control smoothing
        color="blue",
        fill="gray",       # Confidence interval band color
        se=True,
        size=1.2
    )
    + scale_x_continuous(breaks=range(0, 261, 32))
    + labs(
        title="NFL Draft Picks: Career Approximate Value (AV) vs. Overall Pick",
        subtitle="Draft Classes 2015–2021 (Rounds 1–7), span=0.15",
        x="Overall Pick Number",
        y="Career Approximate Value (AV)",
        caption="Data: nflverse / Pro-Football-Reference"
    )
    + theme_minimal(base_size=13)
    + theme(
        plot_title=element_text(weight="bold"),
        panel_grid_minor=element_blank()
    )
)

### Splines
Splines are a class of functions used in regression for splicing lines (splines) together.
The idea is that you create chunks of the range of your predictor and you place knots every so often.  For example you might put a knot at the end of each draft round.  The splines then fit curves (usually cubic polynomials) between each knot and those fitted curves are forced to be smooth at the knots.   You can define them or have the code use the default for knot placement.  The default placement will put the knots

In [ ]:
# Here I define knots for the end of the rounds, every 32 picks.
knots_fit = [32, 64, 96, 128, 160, 192, 224]
fit_spline = smf.ols('dr_av ~ cr(pick, knots=knots_fit)', data=draft_picks).fit()

# fit_spline.summary()

pick_df = pd.DataFrame({'pick': np.arange(1, 257)})
pred_spline = pd.DataFrame({
    'pick': np.arange(1, 257),
    'dr_av': fit_spline.predict(pick_df)
})

# Plot with plotnine using statutory spline smoothing
(
    ggplot(draft_picks, aes(x='pick', y='dr_av'))
    + geom_point(alpha=0.3, color="#2b5c8f")
    + stat_smooth(
        method="lm",
        formula="y ~ cr(x, df=4)",
        color="#e41a1c",
        size=1.2,
        se=True
    )
    + scale_x_continuous(breaks=range(0, 261, 32))
    + labs(
        title="NFL Draft Pick Value: Spline Regression",
        subtitle="Predicting Draft Team AV (dr_av) from Overall Pick Number",
        x="Overall Pick Number",
        y="Draft Team Approximate Value (dr_av)"
    )
    + theme_minimal(base_size=13)
)

Change the knots to be values that might better reflect where there are changes in the values

In [ ]:
knots_fit_custom = [5, 10, 32, 64, 96, 128, 160, 192]
fit_spline = smf.ols('dr_av ~ cr(pick, knots=knots_fit_custom)', data=draft_picks).fit()

# fit_spline.summary()

pred_spline = pd.DataFrame({
    'pick': np.arange(1, 257),
    'dr_av': fit_spline.predict(pick_df)
})

# Plot with plotnine using statutory spline smoothing
(
    ggplot(draft_picks, aes(x='pick', y='dr_av'))
    + geom_point(alpha=0.3, color="#2b5c8f")
    + stat_smooth(
        method="lm",
        formula="y ~ cr(x, df=4)",
        color="#e41a1c",
        size=1.2,
        se=True
    )
    + scale_x_continuous(breaks=range(0, 261, 32))
    + labs(
        title="NFL Draft Pick Value: Spline Regression",
        subtitle="Predicting Draft Team AV (dr_av) from Overall Pick Number",
        x="Overall Pick Number",
        y="Draft Team Approximate Value (dr_av)"
    )
    + theme_minimal(base_size=13)
)

### Jimmy Johnson DVPC

In [ ]:
the_chart = pd.read_csv("https://webpages.charlotte.edu/mschuck1/sports/JimmyJohnsonPickValueChart.csv")

g = (
    ggplot(the_chart, aes(x='Overall_Pick', y='Value'))
    + geom_line(alpha=0.3, color="#2b5c8f")
    + labs(
        title="Jimmy Johnson Draft Pick Value Chart",
        x="Overall Pick Number",
        y="Draft Value"
    )
    + theme_minimal(base_size=13)
)
g

One challenge is to figure out the relative values for the Jimmy Johnson DVPC vs another chart.  There are a couple ways to do this.  First, you can make the totals sum to the same quantity. Second, you can make the first value have the same value.

So here is the two *raw* DVPC values plotted next to each other.

In [ ]:
the_chart = the_chart.rename(columns={'Overall_Pick': 'pick', 'Value': 'JJ_value'})

df_combined = pd.concat([
    the_chart.drop(columns=['pick']).reset_index(drop=True),
    pred_spline.iloc[:224].reset_index(drop=True)
], axis=1)

g = (
    ggplot(df_combined, aes(x='pick', y='JJ_value'))
    + geom_line(alpha=0.3, color="#2b5c8f")
    + geom_line(aes(x='pick', y='dr_av'), alpha=0.4, color="red")
    + labs(
        title="Jimmy Johnson Draft Pick Value Chart",
        x="Overall Pick Number",
        y="Draft Value"
    )
    + theme_minimal(base_size=13)
)
g

So let's first rescale so the sums are the same.

In [ ]:
df_combined['dr_av2'] = df_combined['dr_av'] * (df_combined['JJ_value'].sum() / df_combined['dr_av'].sum())

g = (
    ggplot(df_combined, aes(x='pick', y='JJ_value'))
    + geom_line(alpha=0.3, color="#2b5c8f")
    + geom_line(aes(x='pick', y='dr_av2'), alpha=0.4, color="red")
    + labs(
        title="Jimmy Johnson Draft Pick Value Chart",
        subtitle="Sums of the two curves are the same",
        x="Overall Pick Number",
        y="Draft Value"
    )
    + theme_minimal(base_size=13)
)
g

In [ ]:
df_combined['dr_av3'] = df_combined['dr_av'] * (df_combined['JJ_value'].iloc[0] / df_combined['dr_av'].iloc[0])

g = (
    ggplot(df_combined, aes(x='pick', y='JJ_value'))
    + geom_line(alpha=0.3, color="#2b5c8f")
    + geom_line(aes(x='pick', y='dr_av3'), alpha=0.4, color="red")
    + labs(
        title="Jimmy Johnson Draft Pick Value Chart",
        subtitle="Maxes of the curves are the same",
        x="Overall Pick Number",
        y="Draft Value"
    )
    + theme_minimal(base_size=13)
)
g